In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import io
import numpy as np
import os

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Optimizer
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
import pytorch_lightning as pl
torch.set_float32_matmul_precision('high')

def get_dataloader(dataset_name, batch_size, n_splits):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name.lower() == "mnist":
        dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    elif dataset_name.lower() == "cifar10":
        dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    else:
        raise ValueError(f"Dataset {dataset_name} not supported.")
    
    # Split the dataset into n_splits
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    indices = torch.randperm(dataset_size).tolist()
    split_size = dataset_size // n_splits
    splits = [indices[i * split_size:(i + 1) * split_size] for i in range(n_splits)]
    train_loaders = []
    for i in range(len(splits)):
        splits[i] = torch.utils.data.Subset(dataset, splits[i])
        train_loader = torch.utils.data.DataLoader(splits[0], batch_size=batch_size, shuffle=True)
        train_loaders.append(train_loader)
        
    return train_loaders

In [ ]:
class GradientCollectionModel(pl.LightningModule):
    def __init__(self, model, dataset_name, batch_size):
        super(GradientCollectionModel, self).__init__()
        self.model = model
        self.dataset_name = dataset_name
        self.batch_size = batch_size
        self.current_cycle = {'workerid': None, 'batchid': None, 'flag': True}

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # Unpack batch
        data, target = batch

        # Forward pass
        output = self.forward(data)

        # Compute loss
        loss = nn.CrossEntropyLoss()(output, target)

        self.current_cycle['batchid'] = batch_idx
        self.log('train_loss', loss, on_step=True, on_epoch=False, prog_bar=True)

        return loss

    def on_before_optimizer_step(self, optimizer: Optimizer):
        worker_id = self.current_cycle['workerid']
        batch_idx = self.current_cycle['batchid']
        epoch = self.trainer.current_epoch
        
        if self.current_cycle['flag'][epoch] is not True:
            return super().on_before_optimizer_step(optimizer)

        grad_data = []
        for name, param in self.model.named_parameters():
            if param.grad is None: continue
            grad_data.append(param.grad.cpu().detach().numpy())

        # Now save the gradients to file in parallel using ThreadPoolExecutor
        grad_folder = f'gradients/worker_{worker_id}'
        os.makedirs(grad_folder, exist_ok=True)

        # Define the filename with batch index and epoch
        grad_filename = f'{grad_folder}/grads_epoch_{epoch}_batch_{batch_idx}.npz'

        buf = io.BytesIO()
        np.savez_compressed(buf, *grad_data)
        buf.seek(0)
        with open(grad_filename, 'wb') as f:
            f.write(buf.read())

        return super().on_before_optimizer_step(optimizer)

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=0.0005)


In [ ]:
# Initialize model and trainer
model_name = "custom"
dataset_name = "cifar10"
batch_size = 128

if model_name.lower() == "resnet18":
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
elif model_name.lower() == "custom":
    model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(32 * 16 * 16, 512),
        nn.ReLU(),
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    )
else:
    raise ValueError(f"Model {model_name} not supported.")

# Set up DataLoader
train_loaders = get_dataloader(dataset_name, batch_size, 5)

# Set up PyTorch Lightning model
gradient_model = GradientCollectionModel(model, dataset_name, batch_size)
gradient_model.current_cycle['workerid'] = 1

In [ ]:
# gradient_model.current_cycle['flag'] = [0]*12
# trainer = pl.Trainer(max_epochs=12)
# trainer.fit(gradient_model, train_loaders[0])

In [ ]:
# gradient_model.current_cycle['flag'] = [True,True]
# trainer = pl.Trainer(max_epochs=2)
# trainer.fit(gradient_model, train_loaders[0])

In [ ]:
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#

In [ ]:
# g = []
# g_shapes=[]
# for i in range(2):
#     for j in range(79):
#         with np.load(f'gradients/worker_{1}/grads_epoch_{i}_batch_{j}.npz') as data:
#             grad_data = [data[key].reshape(-1).astype(np.float16) for key in data.files[::2][2:-1]]
#             g_shapes = [data[key].shape for key in data.files[::2][2:-1]]
#             grad_data = np.concatenate(grad_data).astype(np.float16)
#             g.append(grad_data)
# g = np.array(g).astype(np.float16)
# g.shape

In [ ]:
# g_shapes

In [ ]:
# chunk_size = 11111
# cosine_sim = np.zeros((g.shape[1], g.shape[1]), dtype=np.float16)
# 
# for i in range(0, g.shape[1], chunk_size):
#     print(f"{i / g.shape[1] * 100:.2f}%")
#     chunk_end = min(i + chunk_size, g.shape[1])
#     cosine_sim[i:chunk_end, :] = cdist(g[:, i:chunk_end].T, g.T, metric='cosine')
# 
# np.savez_compressed(f'gradients/cosine_similarity_matrix.npz', cosine_sim=cosine_sim)
# 
# cosine_sim = np.load(f'gradients/cosine_similarity_matrix.npz')['cosine_sim']
# for i in range(cosine_sim.shape[0]): cosine_sim[i, i] = 0

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.linalg.blas import dsyrk   # ssyrk if you cast X to float32

def chunked_cor_blocked(X, chunk_size: int = 2_048,
                        dtype_out=np.float32,
                        return_df: bool = False):
    """
    Fast, memory-lean Pearson correlation for tables with
    'many cols / few rows', using two-level blocking.

    Parameters
    ----------
    X : array-like shape (n_samples, n_features)
    chunk_size : int
        Number of feature columns per block.
    dtype_out : np.dtype
        dtype of the returned matrix (float16/32/64).
    return_df : bool
        If True and X is a DataFrame, returns a DataFrame
        with matching row/column labels.

    Returns
    -------
    C : ndarray or DataFrame, shape (n_features, n_features)
    """
    # -- 0. basic prep ----------------------------------------------------
    X = np.asarray(X, dtype=np.float64, order='F')          # F-order aids BLAS
    n_samples, n_features = X.shape

    # centre + scale once, threaded Cython
    X = StandardScaler(with_mean=True, with_std=True,
                       copy=False).fit_transform(X)

    alpha = 1.0 / n_samples                                 # see note below
    C = np.empty((n_features, n_features), dtype=dtype_out)

    # -- 1. blocked double loop ------------------------------------------
    for i in range(0, n_features, chunk_size):
        print(f"{i / n_features * 100:.2f}%")
        i_end = min(i + chunk_size, n_features)
        Xi = X[:, i:i_end]

        # 1.a ⬛ diagonal block (Xiᵀ Xi) via SYRK (upper triangle only)
        blk = dsyrk(alpha=alpha, a=Xi, trans=1, lower=0)

        # symmetrise: copy upper → lower (diag once)
        blk_full = blk + blk.T - np.diag(np.diag(blk))
        C[i:i_end, i:i_end] = blk_full.astype(dtype_out, copy=False)

        # 1.b ▢ off-diagonal blocks (Xiᵀ Xj) via GEMM
        for j in range(i_end, n_features, chunk_size):
            j_end = min(j + chunk_size, n_features)
            Xj = X[:, j:j_end]

            cross = (Xi.T @ Xj) * alpha                     # (k_i × k_j)
            C[i:i_end, j:j_end] = cross.astype(dtype_out, copy=False)
            C[j:j_end, i:i_end] = cross.T.astype(dtype_out, copy=False)

    np.fill_diagonal(C, 1.0)

    if return_df:
        import pandas as pd
        return pd.DataFrame(C,
                            index=getattr(X, "columns", None),
                            columns=getattr(X, "columns", None))
    return C


# sim_matrix=chunked_cor_blocked(g, chunk_size=10000, dtype_out=np.float16)
# np.savez_compressed(f'gradients/corr_similarity_matrix.npz', cosine_sim=sim_matrix)

In [ ]:
sim_matrix = np.load(f'gradients/corr_similarity_matrix.npz')['cosine_sim']

for i in range(sim_matrix.shape[0]): sim_matrix[i, i] = 0
dead_neurons=[]
for i in range(sim_matrix.shape[0]): 
    if np.all(sim_matrix[i] <= 0.00001):
        dead_neurons.append(i)
dead_neurons=np.array(dead_neurons)
sim_matrix = np.delete(np.delete(sim_matrix, dead_neurons, axis=0), dead_neurons, axis=1)
print(sim_matrix.shape)

In [ ]:
plt.hist(np.abs(sim_matrix.flatten()), bins=100)
plt.title('Histogram of similarity values')
plt.show()

In [ ]:
def downsample_to_max(sim_mat, batch_size):
    downsampled = np.zeros((sim_mat.shape[0] // batch_size, sim_mat.shape[1] // batch_size), dtype=np.float16)
    for l1, i in enumerate(range(0, sim_mat.shape[0], batch_size)):
        if i+batch_size > sim_mat.shape[0]: break
        if i % (batch_size*100) == 0: print(f"{i / sim_mat.shape[0] * 100:.2f}%")
        
        for l2, j in enumerate(range(0,sim_mat.shape[1], batch_size)):
            if j+batch_size > sim_mat.shape[1]: break
            
            batch = sim_mat[i:i+batch_size, j:j+batch_size]
            
            batch = np.abs(batch[~np.isnan(batch)])
            downsampled[l1, l2] = (np.max(batch)+np.mean(batch))/2 if batch.size > 0 else 0
    return downsampled

# downsampled_sim_mat=np.round(downsample_to_max(sim_matrix, 10)*256, 0).astype(np.uint8)
# np.savez_compressed(f'gradients/corr_similarity_matrix_downsampled.npz', sim_mat=downsampled_sim_mat)

downsampled_sim_mat = np.load(f'gradients/corr_similarity_matrix_downsampled.npz')['sim_mat']

In [ ]:
plt.figure(figsize=(20, 15), dpi=100)
downsampled_sim_mat[0,0]=255
sns.heatmap(downsampled_sim_mat, cmap='coolwarm', annot=False)
plt.title('corr Similarity Matrix of Gradients')
plt.show()

plt.hist(np.abs(downsampled_sim_mat.flatten()), bins=100)
plt.title('Histogram of similarity values')
plt.show()

In [ ]:
# sort order to have cubes be outputing neurons
sim_matrix = np.load(f'gradients/corr_similarity_matrix.npz')['cosine_sim']

temp=np.concatenate([np.arange(128*512).reshape(128, 512,).T.flatten(),
                     np.arange(128*512, 128*512+64*128).reshape(64, 128,).T.flatten()]).astype(np.int32)
sim_matrix=sim_matrix[temp].T
sim_matrix=sim_matrix[temp].T

In [ ]:
for i in range(sim_matrix.shape[0]): sim_matrix[i, i] = 0
dead_neurons=[]
for i in range(sim_matrix.shape[0]):
    if np.all(sim_matrix[i] <= 0.00001):
        dead_neurons.append(i)
dead_neurons=np.array(dead_neurons)
sim_matrix = np.delete(np.delete(sim_matrix, dead_neurons, axis=0), dead_neurons, axis=1)
print(sim_matrix.shape)

In [ ]:
# downsampled_sim_mat=np.round(downsample_to_max(sim_matrix, 10)*256, 0).astype(np.uint8)
# np.savez_compressed(f'gradients/corr_similarity_matrix_downsampled_out_neuron.npz', sim_mat=downsampled_sim_mat)

downsampled_sim_mat = np.load(f'gradients/corr_similarity_matrix_downsampled_out_neuron.npz')['sim_mat']

In [ ]:
plt.figure(figsize=(20, 15), dpi=100)
downsampled_sim_mat[0,0]=255
sns.heatmap(downsampled_sim_mat, cmap='coolwarm', annot=False)
plt.title('corr Similarity Matrix of Gradients')
plt.show()

plt.hist(np.abs(downsampled_sim_mat.flatten()), bins=100)
plt.title('Histogram of similarity values')
plt.show()

In [ ]:
sim_matrix = np.load(f'gradients/corr_similarity_matrix.npz')['cosine_sim']

for i in range(sim_matrix.shape[0]): sim_matrix[i, i] = 0
dead_neurons = []
for i in range(sim_matrix.shape[0]):
    if np.all(sim_matrix[i] <= 0.00001):
        dead_neurons.append(i)
dead_neurons = np.array(dead_neurons)
sim_matrix = np.delete(np.delete(sim_matrix, dead_neurons, axis=0), dead_neurons, axis=1)
print(sim_matrix.shape)


def downsample_to_max(sim_mat, batch_size):
    downsampled = np.zeros((sim_mat.shape[0] // batch_size, sim_mat.shape[1] // batch_size), dtype=np.float16)
    for l1, i in enumerate(range(0, sim_mat.shape[0], batch_size)):
        if i + batch_size > sim_mat.shape[0]: break
        if i % (batch_size * 100) == 0: print(f"{i / sim_mat.shape[0] * 100:.2f}%")

        for l2, j in enumerate(range(0, sim_mat.shape[1], batch_size)):
            if j + batch_size > sim_mat.shape[1]: break

            batch = sim_mat[i:i + batch_size, j:j + batch_size]

            batch = np.abs(batch[~np.isnan(batch)])
            downsampled[l1, l2] = np.mean(batch) if batch.size > 0 else 0
    return downsampled


downsampled_sim_mat=np.round(downsample_to_max(sim_matrix, 10)*256, 0).astype(np.uint8)
plt.figure(figsize=(20, 15), dpi=100)
downsampled_sim_mat[0,0]=255
sns.heatmap(downsampled_sim_mat, cmap='coolwarm', annot=False)
plt.title('corr Similarity Matrix of Gradients')
plt.show()

plt.hist(np.abs(downsampled_sim_mat.flatten()), bins=100)
plt.title('Histogram of similarity values')
plt.show()




# sort order to have cubes be outputing neurons
sim_matrix = np.load(f'gradients/corr_similarity_matrix.npz')['cosine_sim']

temp = np.concatenate([np.arange(128 * 512).reshape(128, 512, ).T.flatten(),
                       np.arange(128 * 512, 128 * 512 + 64 * 128).reshape(64, 128, ).T.flatten()]).astype(np.int32)
sim_matrix = sim_matrix[temp].T
sim_matrix = sim_matrix[temp].T
for i in range(sim_matrix.shape[0]): sim_matrix[i, i] = 0
dead_neurons = []
for i in range(sim_matrix.shape[0]):
    if np.all(sim_matrix[i] <= 0.00001):
        dead_neurons.append(i)
dead_neurons = np.array(dead_neurons)
sim_matrix = np.delete(np.delete(sim_matrix, dead_neurons, axis=0), dead_neurons, axis=1)
print(sim_matrix.shape)
downsampled_sim_mat=np.round(downsample_to_max(sim_matrix, 10)*256, 0).astype(np.uint8)

plt.figure(figsize=(20, 15), dpi=100)
downsampled_sim_mat[0,0]=255
sns.heatmap(downsampled_sim_mat, cmap='coolwarm', annot=False)
plt.title('corr Similarity Matrix of Gradients')
plt.show()

plt.hist(np.abs(downsampled_sim_mat.flatten()), bins=100)
plt.title('Histogram of similarity values')
plt.show()